# 🧹 02 Data Cleaning Notebook
**วัตถุประสงค์**: ทดลองเขียนฟังก์ชันทำความสะอาดข้อมูล (Data Cleaning) สำหรับทั้ง 3 แหล่งข้อมูล และตรวจสอบผลลัพธ์ก่อนนำไปใส่ใน `src/clean.py`

In [ ]:
import pandas as pd
import numpy as np
import re

PATH_ONE2CAR = '../../01_Raw_Data/one2car/one2car_data.csv'
PATH_US_SALES = '../../01_Raw_Data/us-usecar/used_car_sales.csv'
PATH_SPEC1 = '../../01_Raw_Data/usecar-dataset/used_car_dataset.csv'
PATH_SPEC2 = '../../01_Raw_Data/usecar-dataset/used_cars_dataset_2.csv'

df_one2car = pd.read_csv(PATH_ONE2CAR)
df_us = pd.read_csv(PATH_US_SALES)
df_spec = pd.concat([pd.read_csv(PATH_SPEC1), pd.read_csv(PATH_SPEC2)], ignore_index=True)

print("Datasets Loaded for Cleaning Prototyping!")

--- 
## 🚗 1. Prototype Cleaning Functions for One2Car Data

In [ ]:
# 1.1 ลบแถวที่ไม่มีราคาออกก่อน
df_one2car_clean = df_one2car.dropna(subset=['price']).copy()

# 1.2 ฟังก์ชันแปลงราคาเป็นตัวเลข
def clean_price(val):
    if pd.isna(val):
        return None
    nums = re.sub(r'[^\d]', '', str(val))
    return float(nums) if nums else None

# 1.3 ฟังก์ชันแปลงเลขไมล์
def clean_mileage(val):
    if pd.isna(val):
        return None
    s = str(val).replace('กม.', '').replace(',', '').strip()
    match_range = re.search(r'(\d+)\s*-\s*(\d+)K', s, re.IGNORECASE)
    if match_range:
        low = float(match_range.group(1)) * 1000
        high = float(match_range.group(2)) * 1000
        return int((low + high) / 2)
    nums = re.sub(r'[^\d]', '', s)
    return int(nums) if nums else None

# 1.4 ฟังก์ชันดึง Year, Brand, Model จาก car_title
def parse_car_title(title):
    if pd.isna(title):
        return pd.Series([None, None, None])
    title_str = str(title).strip()
    year_match = re.search(r'^(20\d{2}|19\d{2})', title_str)
    year = int(year_match.group(1)) if year_match else None
    
    text_clean = re.sub(r'^(20\d{2}|19\d{2})\s*', '', title_str)
    parts = text_clean.split()
    brand = parts[0] if len(parts) > 0 else None
    model = parts[1] if len(parts) > 1 else None
    return pd.Series([year, brand, model])

# นำฟังก์ชันไปประยุกต์ใช้
df_one2car_clean['price_clean'] = df_one2car_clean['price'].apply(clean_price)
df_one2car_clean['mileage_clean'] = df_one2car_clean['mileage'].apply(clean_mileage)
df_one2car_clean[['model_year', 'brand', 'model']] = df_one2car_clean['car_title'].apply(parse_car_title)
df_one2car_clean['transmission_clean'] = df_one2car_clean['transmission'].map({
    'เกียร์อัตโนมัติ': 'Automatic',
    'เกียร์ธรรมดา': 'Manual'
}).fillna('Automatic')

df_one2car_clean[['car_title', 'brand', 'model', 'model_year', 'price_clean', 'mileage_clean', 'transmission_clean']].head(10)

--- 
## 🇺🇸 2. Prototype Cleaning for US Sales Dataset

In [ ]:
# 2.1 กรอง Outlier และ Clean คอลัมน์ US Sales
df_us_clean = df_us[(df_us['pricesold'] > 100) & (df_us['Mileage'] > 0)].copy()

# Standardize คอลัมน์
df_us_clean = df_us_clean.rename(columns={
    'pricesold': 'selling_price',
    'yearsold': 'sale_year',
    'Mileage': 'mileage',
    'Make': 'brand',
    'Model': 'model',
    'Year': 'model_year',
    'BodyType': 'body_type'
})

df_us_clean[['selling_price', 'sale_year', 'mileage', 'brand', 'model', 'model_year', 'body_type']].head(10)

--- 
## 🇮🇳 3. Prototype Cleaning for Spec Dataset

In [ ]:
# 3.1 Clean สัญลักษณ์ ₹ และ km ใน Spec dataset
df_spec_clean = df_spec.copy()
df_spec_clean['AskPrice_clean'] = df_spec_clean['AskPrice'].apply(lambda x: float(re.sub(r'[^\d]', '', str(x))) if pd.notna(x) and re.sub(r'[^\d]', '', str(x)) != '' else None)
df_spec_clean['kmDriven_clean'] = df_spec_clean['kmDriven'].apply(lambda x: int(float(re.sub(r'[^\d.]', '', str(x)))) if pd.notna(x) and re.sub(r'[^\d.]', '', str(x)) != '' else None)

df_spec_clean[['Brand', 'model', 'Year', 'kmDriven_clean', 'FuelType', 'Owner', 'AskPrice_clean']].head(10)